In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
from numpy.random import seed, binomial, weibull, exponential, normal
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from relife.lifetime_model import SemiParametricAcceleratedFailureTime

In [ ]:
root_data_path = Path(r"D:\Projets\RTE\ReLife\data")

# Fonctions

In [ ]:
def plot_overlapping_hist_matplotlib(df, bins=30, alpha=0.5):
    """
    Plot overlapping histograms of all numeric columns using matplotlib.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe
    bins : int
        Number of histogram bins
    alpha : float
        Transparency level for overlap
    """
    numeric_df = df.select_dtypes(include="number")

    plt.figure(figsize=(10, 6))

    for col in numeric_df.columns:
        plt.hist(numeric_df[col].dropna(),
                 bins=bins,
                 alpha=alpha,
                 label=col)

    plt.xlabel("Value")
    plt.ylabel("Frequency")
    plt.title("Overlapping Histograms (Matplotlib)")
    plt.legend()
    plt.grid(True)

    plt.show()

In [ ]:
def get_dataset_1(nseed, N): # Weibull shape 1.5, pas de troncature
    seed(nseed)
    covar1 = binomial(n=1, p=0.5, size=N)
    covar2 = normal(scale=0.1, size=N)
    covar = np.concat((covar1[:, None], covar2[:, None]), axis=1)
    params = np.array([1., 2.3])
    g = np.exp(params[0] * covar1 + params[1] * covar2)
    yy = g * weibull(a=1.5, size=N)
    cc = exponential(size=N)
    time = np.minimum(yy, cc)
    event = yy <= cc
    tr = None
    entry = None
    return time, covar, event, entry, yy, cc, tr, params

In [ ]:
def get_dataset_2(nseed, N): # Weibull shape 1.5, avec troncature
    seed(nseed)
    covar1 = binomial(n=1, p=0.5, size=N)
    covar2 = normal(scale=0.1, size=N)
    covar = np.concat((covar1[:, None], covar2[:, None]), axis=1)
    params = np.array([1., 2.3])
    g = np.exp(params[0] * covar1 + params[1] * covar2)
    yy = g * weibull(a=1.5, size=N)
    cc = exponential(size=N)
    time = np.minimum(yy, cc)
    event = yy <= cc
    tr = exponential(scale=0.5, size=N)
    entry = np.minimum(time, tr)
    return time, covar, event, entry, yy, cc, tr, params

In [ ]:
def get_dataset_3(nseed, N): # Weibull shape 1.5, avec troncature plus ancienne
    seed(nseed)
    covar1 = binomial(n=1, p=0.5, size=N)
    covar2 = normal(scale=0.1, size=N)
    covar = np.concat((covar1[:, None], covar2[:, None]), axis=1)
    params = np.array([1., 2.3])
    g = np.exp(params[0] * covar1 + params[1] * covar2)
    yy = g * weibull(a=1.5, size=N)
    cc = exponential(size=N)
    time = np.minimum(yy, cc)
    event = yy <= cc
    tr = exponential(scale=0.25, size=N)
    entry = np.minimum(time, tr)
    return time, covar, event, entry, yy, cc, tr, params

# Récupération des données

## Données d'isolateur

In [ ]:
# Données chaines d'isolateur
relife_csv_datapath = Path(r"D:\Projets\RTE\ReLife\relife\relife\data\csv") # TODO: changer le path en dur
time, event, entry, *args = np.loadtxt(relife_csv_datapath / "insulator_string.csv", delimiter=",", skiprows=1,
                                       unpack=True)
covar = np.column_stack(args)

## Données "Channing"

In [ ]:
# Données Channing
channing_data = pd.read_csv(root_data_path / "channing.csv", sep=";", decimal=",")
channing_data = (
    channing_data
    .drop(columns="time")
    .rename(columns={"exit": "time", "cens": "event"})
)
time, event, entry = channing_data["time"].values, channing_data["event"].astype(float).values, channing_data["entry"].values
covar = (channing_data[["sex"]] == "Male").astype(float).values

## Données simulées

In [ ]:
# Simulating AFT
SEEDS = range(5)
N_RANGE = [100, 500, 1000, 5000]
fun = get_dataset_3

data_sample = {}
for nseed in SEEDS:
    for N in N_RANGE:
        time, covar, event, entry, _, _, _, params = fun(nseed, N)
        data_sample[(nseed, N)] = {
            "io": (time, covar, event, entry),
            "params": params
        }

In [ ]:
# Plot theoretical lifetime and right-censoring time sample distributions
nseed = 4
N = 5000
fun = get_dataset_3

time, _, _, entry, lifetime, right_censoring, left_truncature, _ = fun(nseed, N)

plot_overlapping_hist_matplotlib(
    pd.DataFrame({"time": time, "entry": entry, "time - entry": time - entry})
)

# Fit du modèle

In [ ]:
# Model
model = SemiParametricAcceleratedFailureTime()

## Sur 1 jeu de données

In [ ]:
# Test fit
N = len(covar)

model.fit(
     time=time[:N], covar=covar[:N], event=event[:N], entry=entry[:N] if entry is not None else None
)
print(model.params)

## Sur simulations

In [ ]:
SEEDS = range(5)
N_RANGE = [100, 500, 1000, 5000]

params = {}
for nseed in SEEDS:
    print(f"seed: {nseed}")
    for N in N_RANGE:
        print(f"N: {N}")
        (time, covar, event, entry) = data_sample[(nseed, N)]["io"]
        model.fit(
            time=time, covar=covar, event=event, entry=entry if entry is not None else None
        )
        params[(nseed, N)] = {
            "params_est": model.params,
            "error": data_sample[(nseed, N)]["params"] - model.params
        }

In [ ]:
# Export estimations
set = 3

params_est = pd.DataFrame(params).T["params_est"].apply(pd.Series)
params_est.columns = [f"covar{i}" for i in params_est.columns]
params_est = params_est.reset_index()
columns = list(params_est.columns)
columns[0] = "seed"
columns[1] = "N"
params_est.columns = columns
params_est.to_csv(root_data_path / f"params_est_set{set}.csv", sep=";", decimal=",", index=False)

In [ ]:
# Reimport
set = 3

params_est = pd.read_csv(root_data_path / f"params_est_set{set}.csv", sep=";", decimal=",")

In [ ]:
# Plot
covar_num = 1

sns.barplot(
    data=params_est,
    x="N",
    y=f"covar{covar_num}",
    hue="seed"
)
plt.axhline(y=data_sample[(SEEDS[0], N_RANGE[0])]["params"][covar_num], color="black", linestyle="dashed")